# Notebook 1: Extracción e Inventario de Datos
**Autor:** Leonardo Figueroa, Miguel Flores y Steven Villegas

**Pipeline ETL - NYC Taxi Trip Records**

**Objetivo:** Descargar archivos Parquet de NYC TLC, generar inventario y procesar archivos corruptos de prueba.


## Configuración Inicial
Importar librerías, crear sesión Spark y definir funciones auxiliares.


In [4]:
import sys, os, json, yaml, hashlib, logging, time
from datetime import datetime
from pathlib import Path

# --- MEMORIA Y PYTHON ---
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['PYSPARK_SUBMIT_ARGS'] = '--driver-memory 4g --executor-memory 4g pyspark-shell'
# -------------------------

sys.path.append(os.path.abspath('..'))
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('etl')

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

_config_path = os.path.join(os.path.abspath(".."), "config", "etl_config.yaml")
with open(_config_path, "r", encoding="utf-8") as _f:
    _cfg = yaml.safe_load(_f)
_spark_cfg = _cfg.get("spark", {})
builder = SparkSession.builder \
    .appName("NYC_Taxi_ETL_01") \
    .master(_spark_cfg.get("master", "local[*]"))
for k, v in _spark_cfg.get("config", {}).items():
    builder = builder.config(k, v)
spark = builder.getOrCreate()

logger.info(f"Spark version: {spark.version}")


2026-06-18 17:45:03,965 - etl - INFO - Spark version: 4.1.2


In [2]:
from src.config_loader import *
from src.utils import ensure_dirs, get_headers, generate_process_id, classify_error
from src.extract import download_and_inventory, process_bad_parquet
PROCESS_ID = generate_process_id()
logger.info(f"Process ID: {PROCESS_ID}")
ensure_dirs()


2026-06-18 15:11:12,454 - etl - INFO - Process ID: b5960a1f1f84
2026-06-18 15:11:12,456 - src.utils - INFO - Directory structure verified


---
## FASE 1: Extracción de Datos e Inventario
**Objetivo:** Descargar archivos Parquet desde el repositorio público de NYC TLC y generar un inventario completo con metadatos de cada archivo (tamaño, registros, columnas, hash del esquema).


In [5]:


inventory = download_and_inventory(spark, SERVICES, YEARS)
df_inventory = spark.createDataFrame(inventory)
inventory_path = os.path.join(METADATA_PATH, "audit_file_inventory")
df_inventory.write.mode("overwrite").parquet(inventory_path)
df_inventory_pd = df_inventory.toPandas()
print(f"\n{'='*80}")
print("  INVENTARIO DE ARCHIVOS")
print(f"{'='*80}")
print(f"Total archivos: {len(inventory)}")
print(f"Exitosos: {sum(1 for i in inventory if i['read_status'] == 'SUCCESS')}")
print(f"Fallidos: {sum(1 for i in inventory if i['read_status'] == 'FAILED')}")
print(f"\nArchivos fallidos:")
for i in inventory:
    if i['read_status'] == 'FAILED':
        print(f"  - {i['file_name']}: {i['error_message'][:100]}")
display(df_inventory_pd)


2026-06-18 17:46:02,022 - src.extract - INFO - Read yellow_tripdata_2024-01.parquet: 2964624 records, 19 columns
2026-06-18 17:46:02,316 - src.extract - INFO - Read yellow_tripdata_2024-02.parquet: 3007526 records, 19 columns
2026-06-18 17:46:02,587 - src.extract - INFO - Read yellow_tripdata_2024-03.parquet: 3582628 records, 19 columns
2026-06-18 17:46:02,860 - src.extract - INFO - Read yellow_tripdata_2024-04.parquet: 3514289 records, 19 columns
2026-06-18 17:46:03,129 - src.extract - INFO - Read yellow_tripdata_2024-05.parquet: 3723833 records, 19 columns
2026-06-18 17:46:03,340 - src.extract - INFO - Read yellow_tripdata_2024-06.parquet: 3539193 records, 19 columns
2026-06-18 17:46:03,564 - src.extract - INFO - Read yellow_tripdata_2024-07.parquet: 3076903 records, 19 columns
2026-06-18 17:46:03,759 - src.extract - INFO - Read yellow_tripdata_2024-08.parquet: 2979183 records, 19 columns
2026-06-18 17:46:03,954 - src.extract - INFO - Read yellow_tripdata_2024-09.parquet: 3633030 rec


  INVENTARIO DE ARCHIVOS
Total archivos: 84
Exitosos: 84
Fallidos: 0

Archivos fallidos:


,column_count,error_message,file_name,file_path,file_size_mb,partition_month,partition_year,process_id,processed_at,read_status,record_count,schema_hash,service_type,source_system
0,19,,yellow_tripdata_2024-01.parquet,C:\Users\Usuario-PC\Desktop\Modelado_Avanzado\...,47.65,1,2024,,2026-06-18T17:46:02.022805,SUCCESS,2964624,6069aa5f0abcb1a9,yellow,NYC_TLC
1,19,,yellow_tripdata_2024-02.parquet,C:\Users\Usuario-PC\Desktop\Modelado_Avanzado\...,48.02,2,2024,,2026-06-18T17:46:02.316103,SUCCESS,3007526,6069aa5f0abcb1a9,yellow,NYC_TLC
2,19,,yellow_tripdata_2024-03.parquet,C:\Users\Usuario-PC\Desktop\Modelado_Avanzado\...,57.30,3,2024,,2026-06-18T17:46:02.587360,SUCCESS,3582628,6069aa5f0abcb1a9,yellow,NYC_TLC
3,19,,yellow_tripdata_2024-04.parquet,C:\Users\Usuario-PC\Desktop\Modelado_Avanzado\...,56.39,4,2024,,2026-06-18T17:46:02.860076,SUCCESS,3514289,6069aa5f0abcb1a9,yellow,NYC_TLC
4,19,,yellow_tripdata_2024-05.parquet,C:\Users\Usuario-PC\Desktop\Modelado_Avanzado\...,59.66,5,2024,,2026-06-18T17:46:03.129654,SUCCESS,3723833,6069aa5f0abcb1a9,yellow,NYC_TLC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,25,,fhvhv_tripdata_2025-12.parquet,C:\Users\Usuario-PC\Desktop\Modelado_Avanzado\...,510.94,12,2025,,2026-06-18T17:46:14.318062,SUCCESS,22108438,51fcf4d8a7a50836,fhvhv,NYC_TLC
80,25,,fhvhv_tripdata_2026-01.parquet,C:\Users\Usuario-PC\Desktop\Modelado_Avanzado\...,482.43,1,2026,,2026-06-18T17:46:14.466245,SUCCESS,20940373,51fcf4d8a7a50836,fhvhv,NYC_TLC
81,25,,fhvhv_tripdata_2026-02.parquet,C:\Users\Usuario-PC\Desktop\Modelado_Avanzado\...,454.69,2,2026,,2026-06-18T17:46:14.600284,SUCCESS,19875686,51fcf4d8a7a50836,fhvhv,NYC_TLC
82,25,,fhvhv_tripdata_2026-03.parquet,C:\Users\Usuario-PC\Desktop\Modelado_Avanzado\...,507.94,3,2026,,2026-06-18T17:46:14.768261,SUCCESS,22058358,51fcf4d8a7a50836,fhvhv,NYC_TLC


---
## FASE 1b: Procesamiento de Archivos Parquet Dañados (bad_parquet)
**Objetivo:** Demostrar la detección y clasificación de archivos corruptos desde el repositorio de pruebas.


In [3]:
# Pasar process_id a la función (nueva firma con clasificación enriquecida)
bad_inventory, bad_quarantine = process_bad_parquet(spark, process_id=PROCESS_ID)

if bad_quarantine:
    # Crear DataFrame con todos los campos requeridos
    df_bad_q = spark.createDataFrame(bad_quarantine)
    logger.info(f"Bad parquet quarantine: {len(bad_quarantine)} registros")
    
    print("\n=== ARCHIVOS EN CUARENTENA: CLASIFICACION ===")
    print("Cada archivo dañado fue clasificado según la categoría de recuperación:\n")
    df_bad_q.select("source_file", "rejection_rule", "rejection_stage", 
                    "rejection_column", "technical_reason", "business_reason").show(truncate=False)
    
    print("\n=== DISTRIBUCIÓN POR CATEGORÍA ===")
    df_bad_q.groupBy("rejection_rule").count().orderBy("count", ascending=False).show()
    
    df_bad_q.write.mode("overwrite").parquet(os.path.join(QUARANTINE_PATH, "quarantine_log"))
    logger.info(f"Quarantine log sobrescrito con {len(bad_quarantine)} registros")
    
    if bad_inventory:
        df_bad_inv = spark.createDataFrame(bad_inventory)
        bad_count = df_bad_inv.count()
        logger.info(f"Bad parquet inventario: {bad_count} archivos")
        
        # Fusionar con el inventario principal usando directorio temporal
        audit_path = os.path.join(METADATA_PATH, "audit_file_inventory")
        temp_audit_path = os.path.join(METADATA_PATH, "audit_file_inventory_tmp")
        import shutil
        if os.path.exists(audit_path):
            parquet_files = [f for f in os.listdir(audit_path) if f.endswith(".parquet")]
            if parquet_files:
                df_existing = spark.read.parquet(audit_path)
                df_merged = df_existing.unionByName(df_bad_inv, allowMissingColumns=True)
            else:
                df_merged = df_bad_inv
            if os.path.exists(temp_audit_path):
                shutil.rmtree(temp_audit_path)
            df_merged.write.mode("overwrite").parquet(temp_audit_path)
            if os.path.exists(audit_path):
                shutil.rmtree(audit_path)
            os.rename(temp_audit_path, audit_path)
            logger.info(f"audit_file_inventory actualizado con {bad_count} registros bad_parquet")
        else:
            df_bad_inv.write.mode("overwrite").parquet(audit_path)
else:
    logger.info("No bad_parquet files to process")


2026-06-18 15:12:09,806 - src.extract - WARNING - Bad parquet ARROW-GH-41317.parquet failed as expected: [PARQUET_TYPE_ILLEGAL] Illegal Parquet type: INT32 (TIME(MILLIS,true)). SQLSTATE: 42846
2026-06-18 15:12:09,906 - src.extract - WARNING - Bad parquet ARROW-GH-41321.parquet failed as expected: [PARQUET_TYPE_ILLEGAL] Illegal Parquet type: INT32 (TIME(MILLIS,true)). SQLSTATE: 42846
2026-06-18 15:12:11,931 - src.extract - INFO - Bad parquet ARROW-GH-43605.parquet was actually readable! 21186 records
2026-06-18 15:12:12,123 - src.extract - INFO - Bad parquet ARROW-GH-45185.parquet was actually readable! 5 records
2026-06-18 15:12:12,334 - src.extract - INFO - Bad parquet ARROW-GH-47662.parquet was actually readable! 1000 records
2026-06-18 15:12:12,531 - src.extract - INFO - Bad parquet ARROW-RS-GH-6229-DICTHEADER.parquet was actually readable! 0 records
2026-06-18 15:12:12,692 - src.extract - INFO - Bad parquet ARROW-RS-GH-6229-LEVELS.parquet was actually readable! 1 records
2026-06-18


=== ARCHIVOS EN CUARENTENA: CLASIFICACION ===
Cada archivo dañado fue clasificado según la categoría de recuperación:

+----------------------+---------------------------------------------+------------------------+----------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------+
|source_file           |rejection_rule                               |rejection_stage         |rejection_column|technical_reason                                                                                                                                                                                                                 

2026-06-18 15:12:41,539 - etl - INFO - Quarantine log sobrescrito con 3 registros
2026-06-18 15:12:50,847 - etl - INFO - Bad parquet inventario: 8 archivos
2026-06-18 15:13:00,453 - etl - INFO - audit_file_inventory actualizado con 8 registros bad_parquet


---
## Resumen de Extracción
Pipeline completado. Los archivos se encuentran en data/raw/ y el inventario en data/audit/audit_file_inventory/.
